In [1]:
# ==================================================================================================
# CELL 1 — IMPORT REQUIRED LIBRARIES AND CONFIGURE REPRODUCIBILITY
# ==================================================================================================

import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# --------------------------------------------------------------------------------------------------
# REPRODUCIBILITY
# --------------------------------------------------------------------------------------------------

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


print("=" * 100)
print("PHASE 03.02 — BASELINE CROSS-MODAL MODEL")
print("=" * 100)

print("\nLIBRARIES LOADED")
print("-" * 100)

print(f"NumPy version  : {np.__version__}")
print(f"Pandas version : {pd.__version__}")
print(f"PyTorch version: {torch.__version__}")

print("\nREPRODUCIBILITY")
print("-" * 100)

print(f"Random seed: {RANDOM_SEED}")

print("\nCUDA AVAILABLE:", torch.cuda.is_available())

print("\n" + "=" * 100)
print("✓ ENVIRONMENT INITIALIZED")
print("=" * 100)

PHASE 03.02 — BASELINE CROSS-MODAL MODEL

LIBRARIES LOADED
----------------------------------------------------------------------------------------------------
NumPy version  : 2.5.2
Pandas version : 2.3.3
PyTorch version: 2.13.0+cpu

REPRODUCIBILITY
----------------------------------------------------------------------------------------------------
Random seed: 42

CUDA AVAILABLE: False

✓ ENVIRONMENT INITIALIZED


In [2]:
# ==================================================================================================
# CELL 2 — LOCATE PROJECT ROOT
# ==================================================================================================

def find_project_root(start_path):
    """
    Searches upward from the current directory until the
    GAN-Enhanced-Multimodal-ASD-Screening project structure is found.
    """

    current_path = Path(start_path).resolve()

    for candidate in [current_path] + list(current_path.parents):

        expected_folder = candidate / "06_cross_modal_learning"

        if expected_folder.exists():
            return candidate

    return None


PROJECT_ROOT = find_project_root(Path.cwd())

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "PROJECT_ROOT could not be detected. "
        "Make sure this notebook is inside the project repository."
    )


print("=" * 100)
print("PROJECT ROOT DETECTION")
print("=" * 100)

print("\nPROJECT ROOT:")
print(PROJECT_ROOT)

print("\nCross-modal learning directory:")
print(PROJECT_ROOT / "06_cross_modal_learning")

print("\n" + "=" * 100)
print("✓ PROJECT ROOT DETECTED")
print("=" * 100)

PROJECT ROOT DETECTION

PROJECT ROOT:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

Cross-modal learning directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning

✓ PROJECT ROOT DETECTED


In [3]:
# ==================================================================================================
# CELL 3 — DEFINE REPRESENTATION-AWARE DATASET PATHS
# ==================================================================================================

PAIR_DATA_DIR = (
    PROJECT_ROOT /
    "06_cross_modal_learning" /
    "02_cross_modal_pairing" /
    "pairs" /
    "representation_aware_split"
)


TRAIN_PAIR_PATH = (
    PAIR_DATA_DIR /
    "02_08_train_representation_disjoint_pairs.npz"
)

VALIDATION_PAIR_PATH = (
    PAIR_DATA_DIR /
    "02_08_validation_representation_disjoint_pairs.npz"
)


TRAIN_METADATA_PATH = (
    PAIR_DATA_DIR /
    "02_08_train_representation_disjoint_metadata.csv"
)

VALIDATION_METADATA_PATH = (
    PAIR_DATA_DIR /
    "02_08_validation_representation_disjoint_metadata.csv"
)


print("=" * 100)
print("PHASE 02.08 ARTIFACT PATHS")
print("=" * 100)

artifacts = {
    "Training Pair Dataset": TRAIN_PAIR_PATH,
    "Validation Pair Dataset": VALIDATION_PAIR_PATH,
    "Training Metadata": TRAIN_METADATA_PATH,
    "Validation Metadata": VALIDATION_METADATA_PATH
}

for artifact_name, artifact_path in artifacts.items():

    print(f"\n{artifact_name}")
    print("-" * 100)
    print("Path  :", artifact_path)
    print("Exists:", artifact_path.exists())


missing_artifacts = [
    name
    for name, path in artifacts.items()
    if not path.exists()
]


if missing_artifacts:
    raise FileNotFoundError(
        f"Missing required artifacts: {missing_artifacts}"
    )


print("\n" + "=" * 100)
print("✓ ALL PHASE 02.08 ARTIFACTS FOUND")
print("=" * 100)

PHASE 02.08 ARTIFACT PATHS

Training Pair Dataset
----------------------------------------------------------------------------------------------------
Path  : C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_train_representation_disjoint_pairs.npz
Exists: True

Validation Pair Dataset
----------------------------------------------------------------------------------------------------
Path  : C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_validation_representation_disjoint_pairs.npz
Exists: True

Training Metadata
----------------------------------------------------------------------------------------------------
Path  : C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_

In [4]:
# ==================================================================================================
# CELL 4 — LOAD REPRESENTATION-AWARE CROSS-MODAL PAIRS
# ==================================================================================================

print("=" * 100)
print("LOADING CROSS-MODAL TRAINING AND VALIDATION DATA")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# LOAD TRAINING DATA
# --------------------------------------------------------------------------------------------------

train_data = np.load(TRAIN_PAIR_PATH)

X_source_train = train_data["source_embeddings"].astype(np.float32)
X_target_train = train_data["target_embeddings"].astype(np.float32)
y_train = train_data["pair_labels"].astype(np.float32)


# --------------------------------------------------------------------------------------------------
# LOAD VALIDATION DATA
# --------------------------------------------------------------------------------------------------

validation_data = np.load(VALIDATION_PAIR_PATH)

X_source_val = validation_data["source_embeddings"].astype(np.float32)
X_target_val = validation_data["target_embeddings"].astype(np.float32)
y_val = validation_data["pair_labels"].astype(np.float32)


# --------------------------------------------------------------------------------------------------
# DISPLAY STRUCTURE
# --------------------------------------------------------------------------------------------------

print("\nTRAINING DATA")
print("-" * 100)

print("Source embeddings :", X_source_train.shape)
print("Target embeddings :", X_target_train.shape)
print("Pair labels       :", y_train.shape)


print("\nVALIDATION DATA")
print("-" * 100)

print("Source embeddings :", X_source_val.shape)
print("Target embeddings :", X_target_val.shape)


print("\n" + "=" * 100)
print("✓ CROSS-MODAL PAIR DATA LOADED")
print("=" * 100)

LOADING CROSS-MODAL TRAINING AND VALIDATION DATA

TRAINING DATA
----------------------------------------------------------------------------------------------------
Source embeddings : (1776, 32)
Target embeddings : (1776, 32)
Pair labels       : (1776,)

VALIDATION DATA
----------------------------------------------------------------------------------------------------
Source embeddings : (448, 32)
Target embeddings : (448, 32)

✓ CROSS-MODAL PAIR DATA LOADED


In [5]:
# ==================================================================================================
# CELL 5 — VALIDATE BASELINE MODEL INPUT STRUCTURE
# ==================================================================================================

print("=" * 100)
print("VALIDATING BASELINE MODEL INPUT STRUCTURE")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# TRAINING STRUCTURE
# --------------------------------------------------------------------------------------------------

assert X_source_train.shape == X_target_train.shape, (
    "Training source and target shapes do not match."
)

assert len(X_source_train) == len(y_train), (
    "Training source-label correspondence mismatch."
)


# --------------------------------------------------------------------------------------------------
# VALIDATION STRUCTURE
# --------------------------------------------------------------------------------------------------

assert X_source_val.shape == X_target_val.shape, (
    "Validation source and target shapes do not match."
)

assert len(X_source_val) == len(y_val), (
    "Validation source-label correspondence mismatch."
)


# --------------------------------------------------------------------------------------------------
# EMBEDDING DIMENSION
# --------------------------------------------------------------------------------------------------

assert X_source_train.shape[1] == 32
assert X_target_train.shape[1] == 32

assert X_source_val.shape[1] == 32
assert X_target_val.shape[1] == 32


# --------------------------------------------------------------------------------------------------
# LABEL VALIDATION
# --------------------------------------------------------------------------------------------------

assert set(np.unique(y_train)).issubset({0.0, 1.0})
assert set(np.unique(y_val)).issubset({0.0, 1.0})


print("\nTRAINING")
print("-" * 100)
print("Source shape :", X_source_train.shape)
print("Target shape :", X_target_train.shape)
print("Labels shape :", y_train.shape)


print("\nVALIDATION")
print("-" * 100)
print("Source shape :", X_source_val.shape)
print("Target shape :", X_target_val.shape)
print("Labels shape :", y_val.shape)


print("\nMODEL INPUT DIMENSION")
print("-" * 100)
print("Embedding dimension:", X_source_train.shape[1])


print("\n" + "=" * 100)
print("✓ INPUT STRUCTURE VERIFIED")
print("✓ SOURCE-TARGET CORRESPONDENCE VERIFIED")
print("✓ BINARY RELATIONSHIP LABELS VERIFIED")
print("✓ ALL EMBEDDINGS CONFIRMED AS 32D")
print("=" * 100)

VALIDATING BASELINE MODEL INPUT STRUCTURE

TRAINING
----------------------------------------------------------------------------------------------------
Source shape : (1776, 32)
Target shape : (1776, 32)
Labels shape : (1776,)

VALIDATION
----------------------------------------------------------------------------------------------------
Source shape : (448, 32)
Target shape : (448, 32)
Labels shape : (448,)

MODEL INPUT DIMENSION
----------------------------------------------------------------------------------------------------
Embedding dimension: 32

✓ INPUT STRUCTURE VERIFIED
✓ SOURCE-TARGET CORRESPONDENCE VERIFIED
✓ BINARY RELATIONSHIP LABELS VERIFIED
✓ ALL EMBEDDINGS CONFIRMED AS 32D


In [6]:
# ==================================================================================================
# CELL 6 — LOAD PAIR METADATA FOR CROSS-MODAL CONTEXT
# ==================================================================================================

print("=" * 100)
print("LOADING CROSS-MODAL PAIR METADATA")
print("=" * 100)


train_metadata = pd.read_csv(TRAIN_METADATA_PATH)
validation_metadata = pd.read_csv(VALIDATION_METADATA_PATH)


print("\nTRAINING METADATA")
print("-" * 100)

print("Shape:", train_metadata.shape)

print("\nColumns:")
for column in train_metadata.columns:
    print(" -", column)


print("\nVALIDATION METADATA")
print("-" * 100)

print("Shape:", validation_metadata.shape)


# --------------------------------------------------------------------------------------------------
# CORRESPONDENCE CHECK
# --------------------------------------------------------------------------------------------------

assert len(train_metadata) == len(X_source_train)
assert len(validation_metadata) == len(X_source_val)


print("\nDATASET-METADATA CORRESPONDENCE")
print("-" * 100)

print(
    "Training correspondence:",
    len(train_metadata) == len(X_source_train)
)

print(
    "Validation correspondence:",
    len(validation_metadata) == len(X_source_val)
)


print("\n" + "=" * 100)
print("✓ METADATA LOADED")
print("✓ ROW CORRESPONDENCE VERIFIED")
print("=" * 100)

LOADING CROSS-MODAL PAIR METADATA

TRAINING METADATA
----------------------------------------------------------------------------------------------------
Shape: (1776, 8)

Columns:
 - source_modality
 - target_modality
 - source_index
 - target_index
 - source_class
 - target_class
 - pair_label
 - pair_type

VALIDATION METADATA
----------------------------------------------------------------------------------------------------
Shape: (448, 8)

DATASET-METADATA CORRESPONDENCE
----------------------------------------------------------------------------------------------------
Training correspondence: True
Validation correspondence: True

✓ METADATA LOADED
✓ ROW CORRESPONDENCE VERIFIED


In [7]:
# ==================================================================================================
# CELL 7 — INSPECT MODALITY COMBINATIONS USED BY THE BASELINE
# ==================================================================================================

print("=" * 100)
print("CROSS-MODAL RELATIONSHIP DISTRIBUTION")
print("=" * 100)


def create_modality_summary(metadata, dataset_name):

    summary = (
        metadata
        .groupby(
            ["source_modality", "target_modality", "pair_label"]
        )
        .size()
        .reset_index(name="pair_count")
    )

    print(f"\n{dataset_name}")
    print("-" * 100)

    display(summary)

    return summary


train_modality_summary = create_modality_summary(
    train_metadata,
    "TRAINING MODALITY DISTRIBUTION"
)

validation_modality_summary = create_modality_summary(
    validation_metadata,
    "VALIDATION MODALITY DISTRIBUTION"
)


print("\n" + "=" * 100)
print("✓ CROSS-MODAL MODALITY STRUCTURE INSPECTED")
print("=" * 100)

CROSS-MODAL RELATIONSHIP DISTRIBUTION

TRAINING MODALITY DISTRIBUTION
----------------------------------------------------------------------------------------------------


,source_modality,target_modality,pair_label,pair_count
0,Behavioral,Environmental,0,704
1,Behavioral,Environmental,1,704
2,Genetic,Behavioral,0,92
3,Genetic,Behavioral,1,92
4,Genetic,Environmental,0,92
5,Genetic,Environmental,1,92



VALIDATION MODALITY DISTRIBUTION
----------------------------------------------------------------------------------------------------


,source_modality,target_modality,pair_label,pair_count
0,Behavioral,Environmental,0,176
1,Behavioral,Environmental,1,176
2,Genetic,Behavioral,0,24
3,Genetic,Behavioral,1,24
4,Genetic,Environmental,0,24
5,Genetic,Environmental,1,24



✓ CROSS-MODAL MODALITY STRUCTURE INSPECTED


In [8]:
# ==================================================================================================
# CELL 8 — DEFINE PYTORCH CROSS-MODAL PAIR DATASET
# ==================================================================================================

class CrossModalPairDataset(Dataset):
    """
    PyTorch Dataset for cross-modal representation pairs.

    Each item contains:
        source_embedding
        target_embedding
        pair_label
    """

    def __init__(
        self,
        source_embeddings,
        target_embeddings,
        pair_labels
    ):

        self.source_embeddings = torch.tensor(
            source_embeddings,
            dtype=torch.float32
        )

        self.target_embeddings = torch.tensor(
            target_embeddings,
            dtype=torch.float32
        )

        self.pair_labels = torch.tensor(
            pair_labels,
            dtype=torch.float32
        )


    def __len__(self):

        return len(self.pair_labels)


    def __getitem__(self, index):

        return (
            self.source_embeddings[index],
            self.target_embeddings[index],
            self.pair_labels[index]
        )


print("=" * 100)
print("PYTORCH CROSS-MODAL DATASET CLASS DEFINED")
print("=" * 100)

print("""
Each dataset sample will return:

1. Source embedding → Tensor [32]
2. Target embedding → Tensor [32]
3. Pair label       → Scalar

The Dataset preserves:

Source[i] ↔ Target[i] ↔ Label[i]
""")

print("=" * 100)
print("✓ DATASET CLASS READY")
print("=" * 100)

PYTORCH CROSS-MODAL DATASET CLASS DEFINED

Each dataset sample will return:

1. Source embedding → Tensor [32]
2. Target embedding → Tensor [32]
3. Pair label       → Scalar

The Dataset preserves:

Source[i] ↔ Target[i] ↔ Label[i]

✓ DATASET CLASS READY


In [9]:
# ==================================================================================================
# CELL 9 — CREATE DATASET OBJECTS AND DATALOADERS
# ==================================================================================================

print("=" * 100)
print("CREATING PYTORCH DATASETS AND DATALOADERS")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# DATASET OBJECTS
# --------------------------------------------------------------------------------------------------

train_dataset = CrossModalPairDataset(
    X_source_train,
    X_target_train,
    y_train
)

validation_dataset = CrossModalPairDataset(
    X_source_val,
    X_target_val,
    y_val
)


# --------------------------------------------------------------------------------------------------
# BATCH CONFIGURATION
# --------------------------------------------------------------------------------------------------

BATCH_SIZE = 64


# --------------------------------------------------------------------------------------------------
# DATALOADERS
# --------------------------------------------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


print("\nDATASET SIZE")
print("-" * 100)

print("Training samples  :", len(train_dataset))
print("Validation samples:", len(validation_dataset))


print("\nDATALOADER CONFIGURATION")
print("-" * 100)

print("Batch size              :", BATCH_SIZE)
print("Training batches        :", len(train_loader))
print("Validation batches      :", len(validation_loader))
print("Training shuffle        :", True)
print("Validation shuffle      :", False)


print("\n" + "=" * 100)
print("✓ DATALOADERS CREATED")
print("=" * 100)

CREATING PYTORCH DATASETS AND DATALOADERS

DATASET SIZE
----------------------------------------------------------------------------------------------------
Training samples  : 1776
Validation samples: 448

DATALOADER CONFIGURATION
----------------------------------------------------------------------------------------------------
Batch size              : 64
Training batches        : 28
Validation batches      : 7
Training shuffle        : True
Validation shuffle      : False

✓ DATALOADERS CREATED


In [10]:
# ==================================================================================================
# CELL 10 — INSPECT ONE CROSS-MODAL TRAINING BATCH
# ==================================================================================================

print("=" * 100)
print("INSPECTING ONE TRAINING BATCH")
print("=" * 100)


source_batch, target_batch, label_batch = next(
    iter(train_loader)
)


print("\nBATCH STRUCTURE")
print("-" * 100)

print("Source batch shape:", source_batch.shape)
print("Target batch shape:", target_batch.shape)
print("Label batch shape :", label_batch.shape)


print("\nLABEL DISTRIBUTION IN THIS BATCH")
print("-" * 100)

print(
    "Positive relationships:",
    int(torch.sum(label_batch == 1).item())
)

print(
    "Negative relationships:",
    int(torch.sum(label_batch == 0).item())
)


assert source_batch.shape[1] == 32
assert target_batch.shape[1] == 32


print("\n" + "=" * 100)
print("✓ BATCH STRUCTURE VERIFIED")
print("✓ READY FOR BASELINE MODEL")
print("=" * 100)

INSPECTING ONE TRAINING BATCH

BATCH STRUCTURE
----------------------------------------------------------------------------------------------------
Source batch shape: torch.Size([64, 32])
Target batch shape: torch.Size([64, 32])
Label batch shape : torch.Size([64])

LABEL DISTRIBUTION IN THIS BATCH
----------------------------------------------------------------------------------------------------
Positive relationships: 37
Negative relationships: 27

✓ BATCH STRUCTURE VERIFIED
✓ READY FOR BASELINE MODEL


In [11]:
# ==================================================================================================
# CELL 11 — DEFINE BASELINE CROSS-MODAL FEATURE FUSION STRATEGY
# ==================================================================================================

EMBEDDING_DIM = 32

CONCAT_FEATURE_DIM = EMBEDDING_DIM
ABS_DIFF_FEATURE_DIM = EMBEDDING_DIM
PRODUCT_FEATURE_DIM = EMBEDDING_DIM


FUSION_DIM = (
    CONCAT_FEATURE_DIM +
    CONCAT_FEATURE_DIM +
    ABS_DIFF_FEATURE_DIM +
    PRODUCT_FEATURE_DIM
)


print("=" * 100)
print("BASELINE CROSS-MODAL FEATURE FUSION DESIGN")
print("=" * 100)

print(f"""
INPUT REPRESENTATIONS

Source embedding:
S ∈ R^{EMBEDDING_DIM}

Target embedding:
T ∈ R^{EMBEDDING_DIM}


RELATIONSHIP FEATURES

1. Source representation
   S
   Dimension = {EMBEDDING_DIM}

2. Target representation
   T
   Dimension = {EMBEDDING_DIM}

3. Absolute difference
   |S - T|
   Dimension = {EMBEDDING_DIM}

4. Element-wise interaction
   S × T
   Dimension = {EMBEDDING_DIM}


FINAL FUSION

Z = [S, T, |S-T|, S×T]

Fusion dimension:

{EMBEDDING_DIM} + {EMBEDDING_DIM} + {EMBEDDING_DIM} + {EMBEDDING_DIM}
=
{FUSION_DIM}
""")

print("=" * 100)
print("✓ BASELINE FEATURE FUSION STRATEGY DEFINED")
print("=" * 100)

BASELINE CROSS-MODAL FEATURE FUSION DESIGN

INPUT REPRESENTATIONS

Source embedding:
S ∈ R^32

Target embedding:
T ∈ R^32


RELATIONSHIP FEATURES

1. Source representation
   S
   Dimension = 32

2. Target representation
   T
   Dimension = 32

3. Absolute difference
   |S - T|
   Dimension = 32

4. Element-wise interaction
   S × T
   Dimension = 32


FINAL FUSION

Z = [S, T, |S-T|, S×T]

Fusion dimension:

32 + 32 + 32 + 32
=
128

✓ BASELINE FEATURE FUSION STRATEGY DEFINED


In [2]:
# ==================================================================================================
# CELL 11A — IMPORT PYTORCH MODULES REQUIRED FOR BASELINE MODEL
# ==================================================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 100)
print("PYTORCH MODULES IMPORTED SUCCESSFULLY")
print("=" * 100)

print(f"PyTorch version: {torch.__version__}")
print("torch              : Ready")
print("torch.nn (nn)      : Ready")
print("torch.nn.functional: Ready")

print("=" * 100)

PYTORCH MODULES IMPORTED SUCCESSFULLY
PyTorch version: 2.13.0+cpu
torch              : Ready
torch.nn (nn)      : Ready
torch.nn.functional: Ready


In [7]:
# ==================================================================================================
# CELL 12 — DEFINE BASELINE CROSS-MODAL RELATIONSHIP MODEL
# ==================================================================================================

class BaselineCrossModalModel(nn.Module):
    """
    Baseline neural network for cross-modal relationship learning.

    Input:
        Source embedding : [batch_size, 32]
        Target embedding : [batch_size, 32]

    Feature fusion:
        [Source,
         Target,
         Absolute Difference,
         Element-wise Product]

    Output:
        Relationship logit : [batch_size]
    """

    def __init__(
        self,
        embedding_dim=32,
        hidden_dim_1=128,
        hidden_dim_2=64,
        dropout=0.20
    ):

        super().__init__()


        # ------------------------------------------------------------------------------------------
        # FEATURE DIMENSION
        # ------------------------------------------------------------------------------------------

        fusion_dim = embedding_dim * 4


        # ------------------------------------------------------------------------------------------
        # BASELINE MLP
        # ------------------------------------------------------------------------------------------

        self.network = nn.Sequential(

            nn.Linear(
                fusion_dim,
                hidden_dim_1
            ),

            nn.ReLU(),

            nn.Dropout(dropout),


            nn.Linear(
                hidden_dim_1,
                hidden_dim_2
            ),

            nn.ReLU(),

            nn.Dropout(dropout),


            nn.Linear(
                hidden_dim_2,
                1
            )
        )


    def forward(
        self,
        source,
        target
    ):

        # ------------------------------------------------------------------------------------------
        # CROSS-MODAL FEATURE CONSTRUCTION
        # ------------------------------------------------------------------------------------------

        absolute_difference = torch.abs(
            source - target
        )

        elementwise_product = (
            source * target
        )


        # ------------------------------------------------------------------------------------------
        # FEATURE FUSION
        # ------------------------------------------------------------------------------------------

        fused_features = torch.cat(
            [
                source,
                target,
                absolute_difference,
                elementwise_product
            ],
            dim=1
        )


        # ------------------------------------------------------------------------------------------
        # RELATIONSHIP PREDICTION
        # ------------------------------------------------------------------------------------------

        relationship_logit = self.network(
            fused_features
        )


        # Remove final dimension:
        # [batch_size, 1] → [batch_size]

        relationship_logit = relationship_logit.squeeze(
            dim=1
        )


        return relationship_logit


print("=" * 100)
print("BASELINE CROSS-MODAL MODEL DEFINED")
print("=" * 100)

print("""
MODEL INPUT:

Source → 32D
Target → 32D

FEATURE FUSION:

[Source, Target, |Source-Target|, Source×Target]

                ↓

128D

                ↓

Linear(128 → 128)

                ↓

ReLU + Dropout

                ↓

Linear(128 → 64)

                ↓

ReLU + Dropout

                ↓

Linear(64 → 1)

                ↓

Relationship Logit
""")

print("=" * 100)
print("✓ BASELINE MODEL ARCHITECTURE READY")
print("=" * 100)

BASELINE CROSS-MODAL MODEL DEFINED

MODEL INPUT:

Source → 32D
Target → 32D

FEATURE FUSION:

[Source, Target, |Source-Target|, Source×Target]

                ↓

128D

                ↓

Linear(128 → 128)

                ↓

ReLU + Dropout

                ↓

Linear(128 → 64)

                ↓

ReLU + Dropout

                ↓

Linear(64 → 1)

                ↓

Relationship Logit

✓ BASELINE MODEL ARCHITECTURE READY


In [8]:
# ==================================================================================================
# CELL 13 — INITIALIZE BASELINE CROSS-MODAL MODEL
# ==================================================================================================

print("=" * 100)
print("INITIALIZING BASELINE CROSS-MODAL MODEL")
print("=" * 100)


BASELINE_CONFIG = {
    "embedding_dim": 32,
    "hidden_dim_1": 128,
    "hidden_dim_2": 64,
    "dropout": 0.20
}


baseline_model = BaselineCrossModalModel(
    embedding_dim=BASELINE_CONFIG["embedding_dim"],
    hidden_dim_1=BASELINE_CONFIG["hidden_dim_1"],
    hidden_dim_2=BASELINE_CONFIG["hidden_dim_2"],
    dropout=BASELINE_CONFIG["dropout"]
)


print("\nMODEL CONFIGURATION")
print("-" * 100)

for parameter, value in BASELINE_CONFIG.items():
    print(f"{parameter:<20}: {value}")


print("\nMODEL ARCHITECTURE")
print("-" * 100)

print(baseline_model)


print("\n" + "=" * 100)
print("✓ BASELINE MODEL INITIALIZED")
print("=" * 100)

INITIALIZING BASELINE CROSS-MODAL MODEL

MODEL CONFIGURATION
----------------------------------------------------------------------------------------------------
embedding_dim       : 32
hidden_dim_1        : 128
hidden_dim_2        : 64
dropout             : 0.2

MODEL ARCHITECTURE
----------------------------------------------------------------------------------------------------
BaselineCrossModalModel(
  (network): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

✓ BASELINE MODEL INITIALIZED


In [9]:
# ==================================================================================================
# CELL 14 — ANALYZE BASELINE MODEL COMPLEXITY
# ==================================================================================================

print("=" * 100)
print("BASELINE MODEL PARAMETER ANALYSIS")
print("=" * 100)


total_parameters = sum(
    parameter.numel()
    for parameter in baseline_model.parameters()
)


trainable_parameters = sum(
    parameter.numel()
    for parameter in baseline_model.parameters()
    if parameter.requires_grad
)


print("\nPARAMETER SUMMARY")
print("-" * 100)

print(f"Total parameters     : {total_parameters:,}")
print(f"Trainable parameters : {trainable_parameters:,}")


print("\nLAYER-WISE PARAMETERS")
print("-" * 100)

for name, parameter in baseline_model.named_parameters():

    print(
        f"{name:<35} "
        f"Shape: {str(tuple(parameter.shape)):<20} "
        f"Parameters: {parameter.numel():,}"
    )


print("\n" + "=" * 100)
print("✓ MODEL COMPLEXITY ANALYZED")
print("=" * 100)

BASELINE MODEL PARAMETER ANALYSIS

PARAMETER SUMMARY
----------------------------------------------------------------------------------------------------
Total parameters     : 24,833
Trainable parameters : 24,833

LAYER-WISE PARAMETERS
----------------------------------------------------------------------------------------------------
network.0.weight                    Shape: (128, 128)           Parameters: 16,384
network.0.bias                      Shape: (128,)               Parameters: 128
network.3.weight                    Shape: (64, 128)            Parameters: 8,192
network.3.bias                      Shape: (64,)                Parameters: 64
network.6.weight                    Shape: (1, 64)              Parameters: 64
network.6.bias                      Shape: (1,)                 Parameters: 1

✓ MODEL COMPLEXITY ANALYZED


In [12]:
# ==================================================================================================
# CELL 14A — LOAD REPRESENTATION-AWARE DATASETS AND CREATE DATALOADERS
# ==================================================================================================

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

print("=" * 100)
print("LOADING REPRESENTATION-AWARE DATASETS FOR BASELINE MODEL")
print("=" * 100)


# ==================================================================================================
# 1. DEFINE PROJECT ROOT
# ==================================================================================================

PROJECT_ROOT = Path(
    r"C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening"
)


# ==================================================================================================
# 2. DEFINE DATASET PATHS
# ==================================================================================================

DATASET_DIR = (
    PROJECT_ROOT /
    "06_cross_modal_learning" /
    "02_cross_modal_pairing" /
    "pairs" /
    "representation_aware_split"
)

TRAIN_DATA_PATH = (
    DATASET_DIR /
    "02_08_train_representation_disjoint_pairs.npz"
)

VALIDATION_DATA_PATH = (
    DATASET_DIR /
    "02_08_validation_representation_disjoint_pairs.npz"
)


print("\nDATASET PATHS")
print("-" * 100)

print("Training dataset:")
print(TRAIN_DATA_PATH)

print("\nValidation dataset:")
print(VALIDATION_DATA_PATH)


# ==================================================================================================
# 3. VERIFY FILES EXIST
# ==================================================================================================

assert TRAIN_DATA_PATH.exists(), (
    f"Training dataset not found:\n{TRAIN_DATA_PATH}"
)

assert VALIDATION_DATA_PATH.exists(), (
    f"Validation dataset not found:\n{VALIDATION_DATA_PATH}"
)

print("\n✓ Training dataset found")
print("✓ Validation dataset found")


# ==================================================================================================
# 4. LOAD NPZ DATASETS
# ==================================================================================================

train_data = np.load(TRAIN_DATA_PATH)
validation_data = np.load(VALIDATION_DATA_PATH)


# ==================================================================================================
# 5. EXTRACT ARRAYS
# ==================================================================================================

X_source_train = train_data["source_embeddings"].astype(np.float32)
X_target_train = train_data["target_embeddings"].astype(np.float32)
y_train = train_data["pair_labels"].astype(np.float32)

X_source_val = validation_data["source_embeddings"].astype(np.float32)
X_target_val = validation_data["target_embeddings"].astype(np.float32)
y_val = validation_data["pair_labels"].astype(np.float32)


print("\nDATASET STRUCTURE")
print("-" * 100)

print("\nTRAINING")
print("Source embeddings :", X_source_train.shape)
print("Target embeddings :", X_target_train.shape)
print("Labels            :", y_train.shape)

print("\nVALIDATION")
print("Source embeddings :", X_source_val.shape)
print("Target embeddings :", X_target_val.shape)
print("Labels            :", y_val.shape)


# ==================================================================================================
# 6. DEFINE CUSTOM PYTORCH DATASET
# ==================================================================================================

class CrossModalPairDataset(Dataset):
    """
    PyTorch dataset for cross-modal embedding pairs.
    """

    def __init__(self, source_embeddings, target_embeddings, labels):

        self.source_embeddings = torch.tensor(
            source_embeddings,
            dtype=torch.float32
        )

        self.target_embeddings = torch.tensor(
            target_embeddings,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.float32
        )


    def __len__(self):

        return len(self.labels)


    def __getitem__(self, index):

        return {
            "source": self.source_embeddings[index],
            "target": self.target_embeddings[index],
            "label": self.labels[index]
        }


# ==================================================================================================
# 7. CREATE DATASET OBJECTS
# ==================================================================================================

train_dataset = CrossModalPairDataset(
    X_source_train,
    X_target_train,
    y_train
)

validation_dataset = CrossModalPairDataset(
    X_source_val,
    X_target_val,
    y_val
)


# ==================================================================================================
# 8. CREATE DATALOADERS
# ==================================================================================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# ==================================================================================================
# 9. CONFIGURE DEVICE
# ==================================================================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print("\nDATALOADER CONFIGURATION")
print("-" * 100)

print("Batch size                :", BATCH_SIZE)
print("Training batches          :", len(train_loader))
print("Validation batches        :", len(validation_loader))
print("Computation device        :", device)


print("\n" + "=" * 100)
print("✓ DATASETS LOADED")
print("✓ PYTORCH DATASETS CREATED")
print("✓ TRAINING DATALOADER READY")
print("✓ VALIDATION DATALOADER READY")
print("✓ DEVICE CONFIGURED")
print("✓ BASELINE MODEL NOTEBOOK IS SELF-CONTAINED")
print("=" * 100)

LOADING REPRESENTATION-AWARE DATASETS FOR BASELINE MODEL

DATASET PATHS
----------------------------------------------------------------------------------------------------
Training dataset:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_train_representation_disjoint_pairs.npz

Validation dataset:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_validation_representation_disjoint_pairs.npz

✓ Training dataset found
✓ Validation dataset found

DATASET STRUCTURE
----------------------------------------------------------------------------------------------------

TRAINING
Source embeddings : (1776, 32)
Target embeddings : (1776, 32)
Labels            : (1776,)

VALIDATION
Source embeddings : (448, 32)
Target embeddings : (448, 32)
Labels            : (448,)

DATALOADER CONFIGU

In [14]:
# ==================================================================================================
# CELL 15 — TEST BASELINE MODEL FORWARD PASS
# ==================================================================================================

print("=" * 100)
print("TESTING BASELINE MODEL FORWARD PASS")
print("=" * 100)


baseline_model.eval()


with torch.no_grad():

    test_logits = baseline_model(
        source_batch,
        target_batch
    )


print("\nINPUT")
print("-" * 100)

print("Source batch:", source_batch.shape)
print("Target batch:", target_batch.shape)


print("\nOUTPUT")
print("-" * 100)

print("Relationship logits:", test_logits.shape)


print("\nFIRST 10 LOGITS")
print("-" * 100)

print(test_logits[:10].numpy())


# --------------------------------------------------------------------------------------------------
# STRUCTURE ASSERTION
# --------------------------------------------------------------------------------------------------

assert test_logits.shape[0] == source_batch.shape[0], (
    "Output batch size does not match input batch size."
)

assert test_logits.ndim == 1, (
    "Expected one relationship logit per pair."
)


print("\n" + "=" * 100)
print("✓ FORWARD PASS SUCCESSFUL")
print("✓ ONE OUTPUT GENERATED PER CROSS-MODAL PAIR")
print("=" * 100)

TESTING BASELINE MODEL FORWARD PASS

INPUT
----------------------------------------------------------------------------------------------------
Source batch: torch.Size([32, 32])
Target batch: torch.Size([32, 32])

OUTPUT
----------------------------------------------------------------------------------------------------
Relationship logits: torch.Size([32])

FIRST 10 LOGITS
----------------------------------------------------------------------------------------------------
[0.12322129 0.09383905 0.12745357 0.2240575  0.12531969 0.1429515
 0.10346369 0.12190536 0.10026168 0.14795369]

✓ FORWARD PASS SUCCESSFUL
✓ ONE OUTPUT GENERATED PER CROSS-MODAL PAIR


In [15]:
# ==================================================================================================
# CELL 16 — VERIFY BASELINE MODEL OUTPUT INTEGRITY
# ==================================================================================================

print("=" * 100)
print("VERIFYING BASELINE MODEL OUTPUT NUMERICAL INTEGRITY")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# NaN CHECK
# --------------------------------------------------------------------------------------------------

nan_count = torch.isnan(test_logits).sum().item()


# --------------------------------------------------------------------------------------------------
# INFINITE VALUE CHECK
# --------------------------------------------------------------------------------------------------

infinite_count = torch.isinf(test_logits).sum().item()


print("\nOUTPUT STATISTICS")
print("-" * 100)

print("Number of outputs :", len(test_logits))
print("NaN values        :", nan_count)
print("Infinite values   :", infinite_count)

print("\nLOGIT RANGE")
print("-" * 100)

print("Minimum:", float(test_logits.min()))
print("Maximum:", float(test_logits.max()))
print("Mean   :", float(test_logits.mean()))


assert nan_count == 0, (
    "NaN values detected in model output."
)

assert infinite_count == 0, (
    "Infinite values detected in model output."
)


print("\n" + "=" * 100)
print("✓ MODEL OUTPUTS ARE NUMERICALLY STABLE")
print("=" * 100)

VERIFYING BASELINE MODEL OUTPUT NUMERICAL INTEGRITY

OUTPUT STATISTICS
----------------------------------------------------------------------------------------------------
Number of outputs : 32
NaN values        : 0
Infinite values   : 0

LOGIT RANGE
----------------------------------------------------------------------------------------------------
Minimum: 0.041020467877388
Maximum: 0.22405749559402466
Mean   : 0.1211896538734436

✓ MODEL OUTPUTS ARE NUMERICALLY STABLE


In [17]:
# ==================================================================================================
# CELL 16A — IMPORT PANDAS FOR RESULT INSPECTION
# ==================================================================================================

import pandas as pd

print("=" * 100)
print("PANDAS IMPORTED SUCCESSFULLY")
print("=" * 100)

print(f"Pandas version: {pd.__version__}")

print("\n✓ Ready for prediction result analysis")
print("=" * 100)

PANDAS IMPORTED SUCCESSFULLY
Pandas version: 2.3.3

✓ Ready for prediction result analysis


In [18]:
# ==================================================================================================
# CELL 17 — DEMONSTRATE RELATIONSHIP LOGIT INTERPRETATION
# ==================================================================================================

print("=" * 100)
print("BASELINE OUTPUT INTERPRETATION")
print("=" * 100)


with torch.no_grad():

    relationship_probabilities = torch.sigmoid(
        test_logits
    )


print("\nFIRST 10 PREDICTIONS")
print("-" * 100)

prediction_preview = pd.DataFrame({
    "Logit": test_logits[:10].numpy(),
    "Probability": relationship_probabilities[:10].numpy(),
    "True_Label": label_batch[:10].numpy()
})

display(prediction_preview)


print("""
INTERPRETATION

The model currently produces random initial predictions because:

✓ Architecture has been initialized
✗ Model has NOT been trained yet

A probability near:

1.0 → Model considers relationship likely positive
0.0 → Model considers relationship likely negative
0.5 → Model is uncertain

IMPORTANT:

These values are NOT performance results.

They only demonstrate that the architecture can transform
cross-modal pairs into relationship predictions.

Actual learning begins in Phase 03.04.
""")


print("=" * 100)
print("✓ OUTPUT INTERPRETATION VERIFIED")
print("=" * 100)

BASELINE OUTPUT INTERPRETATION

FIRST 10 PREDICTIONS
----------------------------------------------------------------------------------------------------


,Logit,Probability,True_Label
0,0.123221,0.530766,0.0
1,0.093839,0.523443,1.0
2,0.127454,0.531820,1.0
3,0.224057,0.555781,1.0
4,0.125320,0.531289,1.0
5,0.142952,0.535677,1.0
6,0.103464,0.525843,1.0
7,0.121905,0.530439,1.0
8,0.100262,0.525044,0.0
9,0.147954,0.536921,1.0



INTERPRETATION

The model currently produces random initial predictions because:

✓ Architecture has been initialized
✗ Model has NOT been trained yet

A probability near:

1.0 → Model considers relationship likely positive
0.0 → Model considers relationship likely negative
0.5 → Model is uncertain

IMPORTANT:

These values are NOT performance results.

They only demonstrate that the architecture can transform
cross-modal pairs into relationship predictions.

Actual learning begins in Phase 03.04.

✓ OUTPUT INTERPRETATION VERIFIED


In [19]:
# ==================================================================================================
# CELL 18 — CONFIGURE COMPUTING DEVICE
# ==================================================================================================

print("=" * 100)
print("CONFIGURING COMPUTING DEVICE")
print("=" * 100)


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


baseline_model = baseline_model.to(DEVICE)


print("\nSELECTED DEVICE:")
print(DEVICE)


if torch.cuda.is_available():

    print("\nGPU INFORMATION")
    print("-" * 100)

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

else:

    print("\nRunning on CPU.")


print("\nMODEL DEVICE:")
print(
    next(baseline_model.parameters()).device
)


print("\n" + "=" * 100)
print("✓ COMPUTING DEVICE CONFIGURED")
print("=" * 100)

CONFIGURING COMPUTING DEVICE

SELECTED DEVICE:
cpu

Running on CPU.

MODEL DEVICE:
cpu

✓ COMPUTING DEVICE CONFIGURED


In [22]:
# ==================================================================================================
# CELL 18A — DEFINE REPRODUCIBILITY CONFIGURATION
# ==================================================================================================

# Required imports for reproducibility
import random
import numpy as np
import torch

# Fixed random seed
RANDOM_SEED = 42

# Python random seed
random.seed(RANDOM_SEED)

# NumPy random seed
np.random.seed(RANDOM_SEED)

# PyTorch CPU random seed
torch.manual_seed(RANDOM_SEED)

# PyTorch GPU random seed (only if CUDA is available)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("=" * 100)
print("REPRODUCIBILITY CONFIGURATION")
print("=" * 100)

print(f"Random Seed : {RANDOM_SEED}")
print(f"CUDA Available : {torch.cuda.is_available()}")

print("\n✓ Python random seed configured")
print("✓ NumPy random seed configured")
print("✓ PyTorch random seed configured")

if torch.cuda.is_available():
    print("✓ PyTorch CUDA random seed configured")

print("\n" + "=" * 100)
print("✓ REPRODUCIBILITY CONFIGURATION COMPLETED")
print("=" * 100)

REPRODUCIBILITY CONFIGURATION
Random Seed : 42
CUDA Available : False

✓ Python random seed configured
✓ NumPy random seed configured
✓ PyTorch random seed configured

✓ REPRODUCIBILITY CONFIGURATION COMPLETED


In [24]:
# ==================================================================================================
# CELL 18B — IMPORT JSON MODULE FOR CONFIGURATION SAVING
# ==================================================================================================

import json

print("=" * 100)
print("JSON MODULE IMPORTED SUCCESSFULLY")
print("=" * 100)

print("✓ Ready to save architecture configuration as JSON")
print("=" * 100)

JSON MODULE IMPORTED SUCCESSFULLY
✓ Ready to save architecture configuration as JSON


In [25]:
# ==================================================================================================
# CELL 19 — SAVE BASELINE MODEL ARCHITECTURE CONFIGURATION
# ==================================================================================================

print("=" * 100)
print("SAVING BASELINE MODEL ARCHITECTURE CONFIGURATION")
print("=" * 100)


BASELINE_MODEL_DIR = (
    PROJECT_ROOT /
    "06_cross_modal_learning" /
    "03_cross_modal_training" /
    "03_02_baseline_cross_modal_model" /
    "artifacts"
)


BASELINE_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


ARCHITECTURE_CONFIG_PATH = (
    BASELINE_MODEL_DIR /
    "baseline_model_architecture_config.json"
)


architecture_record = {
    
    "phase": "03.02",
    
    "model_name": "BaselineCrossModalModel",
    
    "purpose": (
        "Establish a simple cross-modal relationship learning "
        "reference before GAN-based modeling."
    ),
    
    "embedding_dimension": 32,
    
    "feature_fusion": [
        "source_embedding",
        "target_embedding",
        "absolute_difference",
        "elementwise_product"
    ],
    
    "fusion_dimension": 128,
    
    "hidden_layers": [
        128,
        64
    ],
    
    "dropout": 0.20,
    
    "output": "single_relationship_logit",
    
    "total_parameters": int(total_parameters),
    
    "random_seed": RANDOM_SEED
}


with open(
    ARCHITECTURE_CONFIG_PATH,
    "w"
) as file:

    json.dump(
        architecture_record,
        file,
        indent=4
    )


print("\nArchitecture configuration saved:")
print(ARCHITECTURE_CONFIG_PATH)


print("\nSAVED CONFIGURATION")
print("-" * 100)

for key, value in architecture_record.items():
    print(f"{key}: {value}")


print("\n" + "=" * 100)
print("✓ BASELINE ARCHITECTURE CONFIGURATION SAVED")
print("=" * 100)

SAVING BASELINE MODEL ARCHITECTURE CONFIGURATION

Architecture configuration saved:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_training\03_02_baseline_cross_modal_model\artifacts\baseline_model_architecture_config.json

SAVED CONFIGURATION
----------------------------------------------------------------------------------------------------
phase: 03.02
model_name: BaselineCrossModalModel
purpose: Establish a simple cross-modal relationship learning reference before GAN-based modeling.
embedding_dimension: 32
feature_fusion: ['source_embedding', 'target_embedding', 'absolute_difference', 'elementwise_product']
fusion_dimension: 128
hidden_layers: [128, 64]
dropout: 0.2
output: single_relationship_logit
total_parameters: 24833
random_seed: 42

✓ BASELINE ARCHITECTURE CONFIGURATION SAVED


In [26]:
# ==================================================================================================
# CELL 20 — PHASE 03.02 BASELINE ARCHITECTURE SUMMARY
# ==================================================================================================

print("=" * 100)
print("PHASE 03.02 — BASELINE CROSS-MODAL MODEL SUMMARY")
print("=" * 100)


summary_data = {
    
    "Component": [
        "Training Pairs",
        "Validation Pairs",
        "Source Embedding Dimension",
        "Target Embedding Dimension",
        "Feature Fusion Dimension",
        "Hidden Layer 1",
        "Hidden Layer 2",
        "Dropout",
        "Output Dimension",
        "Total Trainable Parameters",
        "Forward Pass",
        "Numerical Stability",
        "Computing Device",
        "Training Status"
    ],
    
    "Value": [
        len(X_source_train),
        len(X_source_val),
        32,
        32,
        128,
        128,
        64,
        0.20,
        1,
        trainable_parameters,
        "Verified",
        "Verified",
        str(DEVICE),
        "Not Yet Trained"
    ]
}


baseline_summary = pd.DataFrame(
    summary_data
)


display(baseline_summary)


print("\n" + "=" * 100)

print("""
BASELINE MODEL DESIGN

INPUT
-----
Source embedding : 32D
Target embedding : 32D

FEATURE FUSION
--------------
[S, T, |S-T|, S×T]

Fusion dimension = 128

NEURAL NETWORK
--------------
128 → 128 → 64 → 1

OUTPUT
------
Single relationship logit

PURPOSE
-------
Establish a simple and interpretable reference model before
introducing GAN-based adversarial learning.

IMPORTANT
---------
No training has occurred in this notebook.

The model architecture has been:

✓ Defined
✓ Initialized
✓ Inspected
✓ Parameter-counted
✓ Forward-pass tested
✓ Numerically validated
✓ Configuration saved

NEXT STEP
---------
PHASE 03.03 — CROSS-MODAL LOSS DESIGN
""")

print("=" * 100)
print("✓ PHASE 03.02 BASELINE MODEL ARCHITECTURE COMPLETED")
print("=" * 100)

PHASE 03.02 — BASELINE CROSS-MODAL MODEL SUMMARY


,Component,Value
0,Training Pairs,1776
1,Validation Pairs,448
2,Source Embedding Dimension,32
3,Target Embedding Dimension,32
4,Feature Fusion Dimension,128
5,Hidden Layer 1,128
6,Hidden Layer 2,64
7,Dropout,0.2
8,Output Dimension,1
9,Total Trainable Parameters,24833




BASELINE MODEL DESIGN

INPUT
-----
Source embedding : 32D
Target embedding : 32D

FEATURE FUSION
--------------
[S, T, |S-T|, S×T]

Fusion dimension = 128

NEURAL NETWORK
--------------
128 → 128 → 64 → 1

OUTPUT
------
Single relationship logit

PURPOSE
-------
Establish a simple and interpretable reference model before
introducing GAN-based adversarial learning.

IMPORTANT
---------
No training has occurred in this notebook.

The model architecture has been:

✓ Defined
✓ Initialized
✓ Inspected
✓ Parameter-counted
✓ Forward-pass tested
✓ Numerically validated
✓ Configuration saved

NEXT STEP
---------
PHASE 03.03 — CROSS-MODAL LOSS DESIGN

✓ PHASE 03.02 BASELINE MODEL ARCHITECTURE COMPLETED


In [27]:
# ==================================================================================================
# CELL 21 — NOTEBOOK CONCLUSION
# ==================================================================================================

print("""
====================================================================================================
CONCLUSION — PHASE 03.02 BASELINE CROSS-MODAL MODEL
====================================================================================================

A simple baseline neural network architecture was successfully developed
for learning relationships between cross-modal representation pairs.

The model receives:

    Source Representation (32D)
                +
    Target Representation (32D)

For each pair, four complementary relationship features are constructed:

    1. Source embedding
    2. Target embedding
    3. Absolute difference |S - T|
    4. Element-wise interaction S × T

These features are combined into a 128-dimensional relationship vector:

    Z = [S, T, |S-T|, S×T]

The baseline neural network processes this representation using:

    128 → 128 → 64 → 1

and produces a single relationship logit for each cross-modal pair.

The architecture intentionally remains simple because its purpose is to
establish a minimum performance reference before introducing the more
complex GAN-based architecture.

No model training was performed during this phase.

The architecture was verified through:

✓ Input structure validation
✓ Dataset and metadata correspondence
✓ Cross-modal modality inspection
✓ PyTorch Dataset construction
✓ DataLoader validation
✓ Feature fusion validation
✓ Model parameter analysis
✓ Forward-pass testing
✓ Output numerical stability verification
✓ Architecture configuration saving

The baseline architecture is now ready for objective and loss-function
definition.

NEXT PHASE:

03.03 — CROSS-MODAL LOSS DESIGN

====================================================================================================
""")


CONCLUSION — PHASE 03.02 BASELINE CROSS-MODAL MODEL

A simple baseline neural network architecture was successfully developed
for learning relationships between cross-modal representation pairs.

The model receives:

    Source Representation (32D)
                +
    Target Representation (32D)

For each pair, four complementary relationship features are constructed:

    1. Source embedding
    2. Target embedding
    3. Absolute difference |S - T|
    4. Element-wise interaction S × T

These features are combined into a 128-dimensional relationship vector:

    Z = [S, T, |S-T|, S×T]

The baseline neural network processes this representation using:

    128 → 128 → 64 → 1

and produces a single relationship logit for each cross-modal pair.

The architecture intentionally remains simple because its purpose is to
establish a minimum performance reference before introducing the more
complex GAN-based architecture.

No model training was performed during this phase.

The architecture